In [1]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings

warnings.filterwarnings("ignore")

import os
import ctypes
import sys

# 1. 先设置 R_HOME
os.environ["R_HOME"] = "/home/pxy/miniconda3/envs/r40/lib/R"

# 2. 【核心黑科技】手动加载 R 的动态库
# 这步操作等同于在终端里设置 LD_LIBRARY_PATH，专门解决 VS Code 找不到库的问题
try:
    # 这是 R 的核心库路径
    libR_path = "/home/pxy/miniconda3/envs/r40/lib/R/lib/libR.so"
    # 强制加载进内存
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("✅ 成功强制加载 libR.so")
except OSError as e:
    print(f"❌ 加载失败: {e}")

# 3. 然后再导入其他包
sys.path.append("..") 

import spCLUE
import rpy2.robjects as robjects
print("R 环境路径:", robjects.r['R.home']()[0])

spCLUE.fix_seed(0)

# 定义DLPFC数据集的12个切片ID
slice_ids = [
    "151507", "151508", "151509", "151510",
    "151669", "151670", "151671", "151672",
    "151673", "151674", "151675", "151676"
]

# 用于存储每个切片的ARI结果
ari_results_kmeans = []
ari_results_mclust = []

# 数据路径（请根据实际情况确认路径是否正确）
data_dir = '/home/pxy/home/pxy/data/DLPFC/st/'

# 【新增】创建保存图片的文件夹
figures_dir = "figures_MoEGCL"
if not os.path.exists(figures_dir):
    os.makedirs(figures_dir)
    print(f"Created directory: {figures_dir}")

print(f"Start processing {len(slice_ids)} slices...")

for sample_name in slice_ids:
    spCLUE.fix_seed(0)
    print(f"\n{'='*20} Processing Sample: {sample_name} {'='*20}")
    
    # 1. 设置簇的数量 (根据DLPFC数据集的已知Ground Truth)
    # 151669-151672 通常只有5层，其他切片为7层
    if sample_name in ["151669", "151670", "151671", "151672"]:
        n_clusters = 5
    else:
        n_clusters = 7
    
    try:
        # 2. 加载数据
        # 使用 read_visium 加载数据，路径拼接逻辑参考原文件
        adata = sc.read_visium(data_dir + sample_name)
        adata.var_names_make_unique()
        
        # 加载元数据 (Ground Truth)
        meta = pd.read_csv(data_dir + sample_name + "/metadata.tsv", sep="\t")
        meta = meta.set_index("barcode")
        adata.obs["Region"] = meta.loc[adata.obs_names, "layer_guess_reordered"]
        
        # 3. 数据预处理与构图
        # 原文件 Cell 6 的逻辑
        adata = spCLUE.preprocess(adata)
        adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
        # g_spatia = spCLUE.prepare_graph(adata, "spatial",n_neighbors=12)
        # g_expr = spCLUE.prepare_graph(adata, "expr", metric="euclidean",n_neighbors=12)
        g_spatia = spCLUE.prepare_graph(adata, "spatial")
        g_expr = spCLUE.prepare_graph(adata, "expr", metric="euclidean")
        graph_dict = {"spatial": g_spatia, "expr":g_expr}
                
        
    #     model = spCLUE.spCLUE_TwoStage(
    #     adata.obsm["X_pca"], 
    #     graph_dict, 
    #     n_clusters=n_clusters,
    #     pretrain_epochs=100,   # 预训练200轮
    #     finetune_epochs=100,   # 训练300轮
    #     gamma=0.5,             # 重构损失权重
    #     beta=0.0,              # 聚类损失权重=0 (关键!)
    #     kappa=0.5,             # 对比损失权重
    #     dim_hidden=32,
    #     freeze_encoder=True,   # 冻结预训练编码器
    #     graph_corr=0.5,
    #     dropout=0.5,
    #     residual_weight=0.3
    # )
        
        model = spCLUE.spCLUE_TwoStage(
            adata.obsm["X_pca"], 
            graph_dict, 
            n_clusters=n_clusters,
            dim_input=200,
            pretrain_epochs=100,   # 预训练200轮
            finetune_epochs=100,   # 训练300轮
            gamma=0.0,             # 重构损失权重
            beta=0.0,              # 聚类损失权重=0 (关键!)
            kappa=1.5,             # 对比损失权重
            theta=2.0,
            dim_hidden=32,
            freeze_encoder=False,   # 冻结预训练编码器
            graph_corr=0.2,
            dropout=0.2,
            gate_bias=3.0,
            residual_weight=0.2
        )
        pred, embed, gated_weights = model.train()
        # 5. 聚类
        # 原文件 Cell 10 的逻辑
        # ========== 聚类 ==========
        adata.obsm["spCLUE_twostage"] = embed
        spCLUE.clustering(adata, n_clusters, key="spCLUE_twostage", refinement=True,cluster_methods='kmeans')

        # ========== 评估 ==========
        adata_filtered = adata[adata.obs.Region.notna()]
        ARI_kmeans = adjusted_rand_score(adata_filtered.obs["Region"], 
                                adata_filtered.obs["kmeans_refined"])
        print(f"\nFinal Kmeans ARI on {sample_name}: {ARI_kmeans:.8f}")
        ari_results_kmeans.append(ARI_kmeans)

        # ========== 聚类 ==========
        adata.obsm["spCLUE_twostage"] = embed
        spCLUE.clustering(adata, n_clusters, key="spCLUE_twostage", refinement=True)

        # ========== 评估 ==========
        adata_filtered = adata[adata.obs.Region.notna()]
        ARI_mclust = adjusted_rand_score(adata_filtered.obs["Region"], 
                                adata_filtered.obs["mclust_refined"])
        # print(f"\nFinal Mclust ARI on {sample_name}: {ARI:.4f}")
        
        # 6. 计算 ARI
        # 原文件 Cell 12 的逻辑
        # 过滤掉 Ground Truth 为 NA 的区域
        # adata_valid = adata[adata.obs.Region.notna()]
        # ARI = adjusted_rand_score(adata_valid.obs["Region"], adata_valid.obs["mclust_refined"])
        
        print(f"Sample {sample_name} ARI: {ARI_mclust:.8f}")
        ari_results_mclust.append(ARI_mclust)

        # 绘图：show=False 防止直接显示，便于后续保存
        adata.obs["spCLUE"] = adata.obs["kmeans_refined"]
        sc.pl.spatial(adata, color=["Region", "spCLUE"],show=False, title=["Manual Annotation", f"Ours (ARI={round(ARI_kmeans, 3)})"])
        
        # 保存路径
        save_path_spatial = os.path.join(figures_dir, f"{sample_name}_spatial.png")
        
        # 保存图片 (bbox_inches='tight' 去除多余白边, dpi=300 保证清晰度)
        plt.savefig(save_path_spatial, bbox_inches='tight', dpi=300)
        
        # 关闭当前图形，释放内存 (在循环中非常重要，否则内存会爆)
        plt.close()
        adata.obsm["embed"] = embed
        sc.pp.neighbors(adata, n_neighbors=15, use_rep="embed")
        sc.tl.umap(adata)

        # 绘图：同时查看 Ground Truth (Region) 和 你的聚类结果 (kmeans_refined)
        # 这样可以直观对比哪些区域分错了
        sc.pl.umap(adata, color=["Region", "kmeans_refined"], 
                title=["Ground Truth", "Ours"],
                show=False)

        # 保存图片
        save_path_umap = os.path.join(figures_dir, f"{sample_name}_umap.png")
        plt.savefig(save_path_umap, bbox_inches='tight', dpi=300)
        plt.close()

        # 计算 PAGA
        # groups 指定聚类结果所在的 obs 列名
        sc.tl.paga(adata, groups="kmeans_refined")

        # 绘图
        # plot_threshold 可以控制显示连通性的阈值
        sc.pl.paga(adata, color="kmeans_refined", 
                title=f"PAGA Graph - {sample_name}",
                show=False)

        # 保存图片
        save_path_paga = os.path.join(figures_dir, f"{sample_name}_paga.png")
        plt.savefig(save_path_paga, bbox_inches='tight')
        plt.close()

        
        # print(f"All Figures saved to: {save_path}")

        import torch

        del model
        del adata
        del embed

        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"Error processing sample {sample_name}: {e}")

# 7. 输出最终统计结果
print(f"\n{'='*20} Final Results {'='*20}")
if ari_results_kmeans:
    mean_ari = np.mean(ari_results_kmeans)
    median_ari = np.median(ari_results_kmeans)
    print(f"ARI per slice: {[round(x, 5) for x in ari_results_kmeans]}")
    print(f"Mean ARI: {mean_ari:.4f}")
    print(f"Median ARI: {median_ari:.4f}")
else:
    print("No ARI results collected.")
    print(f"\n{'='*20} Final Results {'='*20}")
if ari_results_mclust:
    mean_ari = np.mean(ari_results_mclust)
    median_ari = np.median(ari_results_mclust)
    print(f"ARI per slice: {[round(x, 5) for x in ari_results_mclust]}")
    print(f"Mean ARI: {mean_ari:.4f}")
    print(f"Median ARI: {median_ari:.4f}")
else:
    print("No ARI results collected.")

✅ 成功强制加载 libR.so
R 环境路径: /home/pxy/miniconda3/envs/r40/lib/R
Start processing 12 slices...

==================== Processing Sample: 151507 ====================
normalized data ---------------->
正在构建图: spatial, 使用度量: cosine ...
  -> 使用空间坐标 (euclidean)
  -> 计算最近邻 (NearestNeighbors)...
  -> 对称化与归一化...
spatial graph created successfully <----

正在构建图: expr, 使用度量: euclidean ...
  -> 使用 PCA 表达特征
  -> 计算最近邻 (NearestNeighbors)...
  -> 对称化与归一化...
expr graph created successfully <----

Stage 1: Pre-training Shared Encoder


Pretrain:  21%|██        | 21/100 [00:02<00:04, 16.38it/s]

  Pretrain Epoch 10: Rec Loss = 10.313286
  Pretrain Epoch 20: Rec Loss = 10.152754


Pretrain:  42%|████▏     | 42/100 [00:02<00:01, 37.59it/s]

  Pretrain Epoch 30: Rec Loss = 10.004379
  Pretrain Epoch 40: Rec Loss = 9.938392
  Pretrain Epoch 50: Rec Loss = 9.917346


...

  Train Epoch 80: Loss = 12.151836,Rec Loss = 12.028075, Contrast Loss = 8.038137,  Cluster Loss = 0.000000, Smooth Loss = 0.047315


Finetune:  91%|█████████ | 91/100 [00:10<00:00, 10.59it/s]

  Train Epoch 90: Loss = 12.133734,Rec Loss = 12.030672, Contrast Loss = 8.032254,  Cluster Loss = 0.000000, Smooth Loss = 0.042676


Finetune: 100%|██████████| 100/100 [00:10<00:00,  9.10it/s]


  Train Epoch 100: Loss = 12.120803,Rec Loss = 12.032811, Contrast Loss = 8.027666,  Cluster Loss = 0.000000, Smooth Loss = 0.039652

  Finetune Epoch 100:
    Total Loss   = 12.1208
    Rec Loss     = 12.0328
    Contrast Loss = 8.0277
    Smooth Loss = 0.0397
    Gate: spatial=0.999±0.009, expr=0.001±0.001

✓ Finetune finished (max epochs reached)
  [Gate Stats] spatial=0.999±0.009, expr=0.001±0.001

Final Kmeans ARI on 151676: 0.49099983
fitting ...
  |======================================================================| 100%
Sample 151676 ARI: 0.51934549

==================== Final Results ====================
ARI per slice: [0.5682, 0.51792, 0.55731, 0.49471, 0.58417, 0.65953, 0.78057, 0.80392, 0.50411, 0.59707, 0.44326, 0.491]
Mean ARI: 0.5835
Median ARI: 0.5628
ARI per slice: [0.55472, 0.42522, 0.53201, 0.48905, 0.362, 0.32051, 0.51481, 0.81468, 0.43415, 0.54124, 0.43965, 0.51935]
Mean ARI: 0.4956
Median ARI: 0.5019
